En este notebook arme una v2 de mi dataset para mi proyecto de machine learning. Para esta v2 limpié las comillas tanto en los nombres de columnas como en las celdas de texto para que al exportarlo quede un CSV estándar. Transformé cada variable a su debido formato (pasaron de estar todas como object a estar en int, float o datetime según el caso) con el fin de que ya quede limpio para mis próximas entregas. Verifiqué que mi columna de la expiración de los contratos al momento de la transferencia es cualquier cosa, me dí cuenta haciendo una pequeña auditoría estadística en uno de los proyectos anteriores que la varianza de estos registros fue nula y la razón era que todos los contratos tenían un tiempo restante de 24 meses. Más adelante voy a borrar esta columna si no le encuentro una solución. Para mis proximos entornos debo de reever el asunto del proxy para el cálculo del poder de compra de los clubes compradores. Ahora mismo las ponderaciones se rigen por los siguientes multiplicadore: ligue 1 - 1.0; premier league - 1.8; Serie A - 1.1 y Bundes liga - 1.3. Quizas existan otros metodos mas fieles que estos multiplicadores. No lo se, como ya dije, lo dejaré para otra versión del dataset. Por ultimo, en esta v2 Agregué una columna más a mi dataset que pasa a millones de euros los montos de las transferencias con el fin de hacer la lectura más sencilla.

In [12]:
import pandas as pd
import numpy as np
import csv

# 1. Cargamos el consolidado ignorando las comillas problemáticas de origen
df = pd.read_csv("dataset_futbol_consolidado.csv", quoting=csv.QUOTE_NONE)

# 2. Limpiamos las comillas de los nombres de las columnas
df.columns = df.columns.str.replace('"', '', regex=False).str.strip()

# 3. Limpiamos las comillas en las columnas de texto SIN convertir los NaN en string todavía
for col in df.select_dtypes(include=['object']).columns:
    df[col] = df[col].str.replace('"', '', regex=False).str.strip()

print("Columnas limpias:", df.columns.tolist())
display(df.head(3))
display(df.shape)

Columnas limpias: ['player_id', 'player_name', 'transfer_date', 'selling_club_name', 'goals_prior_season', 'assists_prior_season', 'minutes_played_prior_season', 'national_team_caps', 'player_age', 'contract_left_months', 'player_position', 'foot', 'buying_club_league', 'destination_gdp_per_capita', 'buying_club_revenue', 'transfer_fee_eur']


,player_id,player_name,transfer_date,selling_club_name,goals_prior_season,assists_prior_season,minutes_played_prior_season,national_team_caps,player_age,contract_left_months,player_position,foot,buying_club_league,destination_gdp_per_capita,buying_club_revenue,transfer_fee_eur
0,164771,Jordan Veretout,2024-09-04,Marseille,4.0,4.0,3388.0,6.0,31.5,24,Midfield,right,FR1,55441.188,55441.19,4000000.0
1,211474,Ali Abdi,2024-09-02,SM Caen,0.0,0.0,0.0,49.0,30.7,24,Defender,left,FR1,55441.188,55441.19,500000.0
2,110864,Sam Johnstone,2024-08-30,Crystal Palace,0.0,0.0,2048.0,4.0,31.4,24,Goalkeeper,right,GB1,54542.016,98175.63,11900000.0


(5290, 16)

In [14]:
# 1. Lista de columnas que deben ser numéricas
columnas_numericas = [
    'player_id', 'goals_prior_season', 'assists_prior_season',
    'minutes_played_prior_season', 'national_team_caps', 'player_age',
    'contract_left_months', 'destination_gdp_per_capita',
    'buying_club_revenue', 'transfer_fee_eur'
]

for col in columnas_numericas:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# 2. Convertimos la fecha a formato datetime
df['transfer_date'] = pd.to_datetime(df['transfer_date'], errors='coerce')

# 3. Verificamos cómo quedaron los tipos y cuántos nulos reales hay ahora
print("--- TIPOS DE DATOS ---")
print(df.dtypes)

print("\n--- CONTEO DE NULOS REALES ---")
print(df.isnull().sum()[df.isnull().sum() > 0])

--- TIPOS DE DATOS ---
player_id                               int64
player_name                            object
transfer_date                  datetime64[ns]
selling_club_name                      object
goals_prior_season                    float64
assists_prior_season                  float64
minutes_played_prior_season           float64
national_team_caps                    float64
player_age                            float64
contract_left_months                    int64
player_position                        object
foot                                   object
buying_club_league                     object
destination_gdp_per_capita            float64
buying_club_revenue                   float64
transfer_fee_eur                      float64
dtype: object

--- CONTEO DE NULOS REALES ---
national_team_caps    828
player_age              2
foot                    6
dtype: int64


In [20]:
# Cargamos los archivos originales seleccionados para inspeccionar columnas de contrato
df_players_raw = pd.read_csv("players-selected-columns.csv")
df_transfers_raw = pd.read_csv("transfers-selected-columns.csv")

print("Columnas en players:", df_players_raw.columns.tolist())
print("Columnas en transfers:", df_transfers_raw.columns.tolist())

# Buscamos cualquier columna que mencione 'contract' o 'expiration'
cols_contrato = [c for c in df_players_raw.columns if 'contract' in c.lower() or 'expir' in c.lower()]
print("\nColumnas de contrato encontradas en players:", cols_contrato)

if cols_contrato:
    display(df_players_raw[['player_id'] + cols_contrato].dropna().head(5))

Columnas en players: ['player_id', 'name', 'date_of_birth', 'position', 'foot', 'international_caps']
Columnas en transfers: ['player_id', 'transfer_date', 'transfer_season', 'to_club_id', 'from_club_name', 'transfer_fee', 'market_value_in_eur', 'player_name']

Columnas de contrato encontradas en players: []


In [25]:
# Agregamos el monto en millones de euros para facilitar la lectura e interpretación
df['transfer_fee_millions'] = (df['transfer_fee_eur'] / 1_000_000).round(4)

# Re-exportamos el dataset v2 actualizado (ahora con 17 columnas)
df.to_csv("dataset_futbol_v2.csv", index=False)
print("v2 actualizado con transfer_fee_millions. Nuevas dimensiones:", df.shape)
display(df.head())

v2 actualizado con transfer_fee_millions. Nuevas dimensiones: (5290, 17)


,player_id,player_name,transfer_date,selling_club_name,goals_prior_season,assists_prior_season,minutes_played_prior_season,national_team_caps,player_age,contract_left_months,player_position,foot,buying_club_league,destination_gdp_per_capita,buying_club_revenue,transfer_fee_eur,transfer_fee_millions
0,164771,Jordan Veretout,2024-09-04,Marseille,4.0,4.0,3388.0,6.0,31.5,24,Midfield,right,FR1,55441.188,55441.19,4000000.0,4.0
1,211474,Ali Abdi,2024-09-02,SM Caen,0.0,0.0,0.0,49.0,30.7,24,Defender,left,FR1,55441.188,55441.19,500000.0,0.5
2,110864,Sam Johnstone,2024-08-30,Crystal Palace,0.0,0.0,2048.0,4.0,31.4,24,Goalkeeper,right,GB1,54542.016,98175.63,11900000.0,11.9
3,160668,Antonio Colak,2024-08-30,Parma,0.0,0.0,0.0,3.0,31.0,24,Attack,right,IT1,53312.070,58643.28,2500000.0,2.5
4,251302,Niklas Dorsch,2024-08-30,Augsburg,0.0,0.0,1449.0,13.0,26.6,24,Midfield,right,L1,63578.140,82651.58,3500000.0,3.5
